# Zoos and Aquariums — Total Count

This notebook filters the museums dataset for zoos and aquariums and reports the combined total.

Self-contained: only requires `museums.csv` in the same folder. Run **Kernel → Restart Kernel and Run All Cells**.

## Setup

In [1]:
import pandas as pd
df = pd.read_csv("museums.csv", low_memory=False)
print(f"Loaded {len(df):,} museums")

Loaded 33,072 museums


## Step 1: How does the dataset categorize museums?

Look at all `Museum Type` values to understand how zoos and aquariums are classified.

In [2]:
df["Museum Type"].value_counts(dropna=False)

Museum Type
HISTORIC PRESERVATION                            14861
GENERAL MUSEUM                                    8699
ART MUSEUM                                        3241
HISTORY MUSEUM                                    2284
ARBORETUM, BOTANICAL GARDEN, OR NATURE CENTER     1484
SCIENCE & TECHNOLOGY MUSEUM OR PLANETARIUM        1081
ZOO, AQUARIUM, OR WILDLIFE CONSERVATION            564
CHILDREN'S MUSEUM                                  512
NATURAL HISTORY MUSEUM                             346
Name: count, dtype: int64

**Important:** the IMLS dataset bundles zoos, aquariums, and wildlife conservation into a **single combined category** — there is no separate tag for "zoo" vs "aquarium" vs "wildlife preserve" in the source data. So the count we get from a category filter will include all three.

## Step 2: Filter by the IMLS category

In [3]:
category = "ZOO, AQUARIUM, OR WILDLIFE CONSERVATION"
mask = df["Museum Type"] == category
category_subset = df[mask]

category_count = len(category_subset)
print(f"Museums in '{category}': {category_count}")

Museums in 'ZOO, AQUARIUM, OR WILDLIFE CONSERVATION': 564


## Step 3: How many of those are actually zoos or aquariums by name?

Inspect the names to see what's in the bucket beyond zoos and aquariums.

In [4]:
names = category_subset["Museum Name"].dropna()

is_zoo      = names.str.contains(r"\bzoo", case=False, regex=True)
is_aquarium = names.str.contains(r"aquarium", case=False, regex=True)

print(f"In-category total                        : {len(category_subset)}")
print(f"  ...with 'zoo' in name                 : {is_zoo.sum()}")
print(f"  ...with 'aquarium' in name            : {is_aquarium.sum()}")
print(f"  ...with either (= zoos + aquariums)   : {(is_zoo | is_aquarium).sum()}")
print(f"  ...with neither (wildlife/sanctuaries): {(~(is_zoo | is_aquarium)).sum()}")

In-category total                        : 564
  ...with 'zoo' in name                 : 258
  ...with 'aquarium' in name            : 100
  ...with either (= zoos + aquariums)   : 354
  ...with neither (wildlife/sanctuaries): 210


Examples of what's in the bucket but isn't a zoo or aquarium:

In [5]:
neither = names[~(is_zoo | is_aquarium)]
neither.sample(10, random_state=1).tolist()

['BIG CAT HABITAT GULF COAST SANCTUARY',
 'WILDLIFE CORRIDOR',
 'FORT CAROLINE NATIONAL MEMORIAL TIMUCUAN ECOLOGICAL & HISTORIC PRESERVE NATIONAL PARK',
 'FOX VALLEY WILDLIFE MUSEUM',
 'ROCKY MOUNTAIN WILDLIFE MUSEUM UCE',
 'WAQUOIT BAY NATIONAL ESTUARINE RESEARCH RESERVE',
 'BUTTERFLY PAVILION',
 'WILDERNESS WILDLIFE MUSEUM',
 'HUNT HILL AUDUBON SANCTUARY',
 'PRESERVE HISTORIC SAINT MARYS']

## Step 4: Catch zoos and aquariums classified under other categories

A small number of museums named "zoo" or "aquarium" are classified differently in the dataset (e.g., as Children's Museum or Science Museum). Include those for a complete count.

In [6]:
all_names = df["Museum Name"].dropna()
name_mask = all_names.str.contains(r"\bzoo|aquarium", case=False, regex=True)
name_based_total = name_mask.sum()

in_other_categories = name_mask.sum() - (is_zoo | is_aquarium).sum()
print(f"Museums anywhere in dataset with 'zoo' or 'aquarium' in name: {name_based_total}")
print(f"  of which classified in OTHER categories: {in_other_categories}")

Museums anywhere in dataset with 'zoo' or 'aquarium' in name: 385
  of which classified in OTHER categories: 31


## Step 5: Final answer

In [7]:
category_total   = category_count
name_strict_total = name_based_total

print("=" * 60)
print("TOTAL ZOOS AND AQUARIUMS")
print("=" * 60)
print(f"By IMLS category (includes wildlife conservation): {category_total}")
print(f"By name match  ('zoo' or 'aquarium' in name)    : {name_strict_total}")
print("=" * 60)

TOTAL ZOOS AND AQUARIUMS
By IMLS category (includes wildlife conservation): 564
By name match  ('zoo' or 'aquarium' in name)    : 385


## Summary

**Two valid answers depending on definition:**

- **564** if you take the IMLS classification at face value — this is the count of records tagged `ZOO, AQUARIUM, OR WILDLIFE CONSERVATION`. Note that it includes ~210 wildlife sanctuaries, ecological reserves, and butterfly pavilions in addition to actual zoos and aquariums.

- **385** if you require the museum's name to actually contain "zoo" or "aquarium" — a stricter definition that excludes wildlife-conservation entries while picking up 31 zoos/aquariums miscategorized in other buckets (e.g., children's museums or science museums).

The 564 figure is the cleanest single-number answer to "how many zoos and aquariums are in the dataset" because it aligns with how the dataset itself counts them. Use 385 when the strict definition matters (e.g., for visitor-attendance comparisons).